# Complex Networks Library - a quick guide

This personal library was developed to calculate and analyze network properties easier.

At computational scope, it saves time and energy by looking in a hash-table for pre-calculated dependencies (for instance, other measures) and use them instead of computing all again, as some other libraries will do.

At scientific scope, it stores the values on dataframes, which are easy to manipulate. A lot of database-like operations are implemented, as like some statistical measures. Also, it is compatible with plotting libraries.

We use 2 main entities here: the `GraphAPI` as the data structure to be analyzed, and the `DataPool` as a database for collections of networks and measures, with their results (after being properly calculated).

The first one is an encapsulation of common graph objects. As there is no consense about using `igraph` or `networkx` library, this class allow both of them to be under the sheets, acessed through standarized function calls (for simple uses) of directly by the `data` property (for specific uses). Also, simple graphs can be easily converted from one library to another, also is easy to generate common complex network models.

The last one stores the data in the following dataframes: `networks`, where the graphs and their metadata are stored; `measures`, where the functions and dependences are configurated; and the `results`, that is just a key-value table.

In the data pool tables, some columns may seem to be unnecessary at first glance, as there were included to be used with classification purposes. For example, networks' _collection_, _dataset_ and _label_ properties. It can be ignored here, as we do not aim to classify networks.

Important things to know:

- the "graph" object requested in the measure functions is the `GraphAPI` class; but the graph itself is stored in the `data` property (here we use `igraph` as backbone for efficiency);
- the basic pipeline to use the `DataPool` class from this analytical library is to populate the data pool with the set of networks and measures, and then call the `evaluate(.)` method (possibly defining subsets);
- to summarize the results, the best way is to use the `Report.make(.)` static method, defining a scope of measures (graph-level, node-level, ...) and the measures identifier column;
- as some of the measures are list-like or dict-like, is it possible to request the reporting tool to expand them (explore as new columns).

In [ ]:
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd

In [ ]:
import sys
sys.path.insert(0, '../src/network_analysis/lib') 
from containers import GraphAPI
from analytics import DataPool, Report
from characterization import Distances, Connectivity, ClusteringAndCycles, Centrality

# How to calculate measures individually

In [ ]:
# generating with igraph backbone
G1 = GraphAPI.generate(engine='ig', style='ba', n=25, m=3)
G1.draw()

In [ ]:
# generating with networkx backbone
G2 = GraphAPI.generate(engine='nx', style='ba', n=25, m=3)
G2.draw()

We can calculate the dependences and pass to the method, but also just call the method itself without nothing besides the graph and it will calculate the dependences inside.

In [ ]:
q3 = ClusteringAndCycles.qt_triplets(G1) # node-level
cc = ClusteringAndCycles.vertex_clustering_coeff(G1, q3) # node-level
cc

# How to calculate measures in batch

# Retrieving and analising results

In [ ]:
datapool = DataPool()
datapool.load('../data/info', n=True, m=True, r=True)

In [ ]:
datapool.networks

In [ ]:
datapool.measures

In [ ]:
m_mask = (datapool.measures['scope']=='graph')
rep_g = Report.make(datapool, measure_subset=m_mask, header='varname', expand=True, detail=False)
rep_g

In [ ]:
m_mask = (datapool.measures['scope']=='vertex')
rep_v = Report.make(datapool, measure_subset=m_mask, header='varname', expand=True, detail=False)
rep_v

In [ ]:
rep_g['params'] = list(map(lambda args: args[1].split('_')[0], rep_g.index))
rep_v['params'] = list(map(lambda args: args[1].split('_')[0], rep_v.index))

The measures at graph level visibly changes, but not those at node level.

In [ ]:
cols_exc = ['diam', 'avg_deg', 'skw_deg', 'krt_deg', 'hmean_gist', 'effic']
cols_inc = [ c for c in rep_g.columns if c not in cols_exc ]
sns.pairplot(
    data = rep_g[cols_inc],
    hue = 'params',
    corner = True,
)
plt.tight_layout()
plt.show()

In [ ]:
sns.pairplot(
    data = rep_v,
    hue = 'params',
    corner = True,
)
plt.tight_layout()
plt.show()